In [7]:
import pandas as pd
import joblib, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('../data/processed/mental_processed.csv')

question_cols = (
    [f'Q3_{i}_S{i}' for i in range(1, 8)] +
    [f'Q3_{i}_A{i-7}' for i in range(8, 15)] +
    [f'Q3_{i}_D{i-14}' for i in range(15, 22)]
)
print("Question columns:", len(question_cols))   # should be 21

# adjust 'Age' / 'Gender' if your column names differ in case
feature_cols = question_cols + ['Q1_1', 'Q1_2']

X = df[feature_cols].copy()
y = df['mental_risk_category']

print("Feature shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
X = df[feature_cols].copy()
y = df['mental_risk_category']
print(X.shape, y.value_counts())
print("Question columns:", len(question_cols))
print("Demographic columns:", [c for c in df.columns if 'age' in c.lower() or 'gender' in c.lower()])
print("Missing question columns:", [c for c in question_cols if c not in df.columns])

Question columns: 21
Feature shape: (1812, 23)
Target distribution:
mental_risk_category
High        771
Low         562
Moderate    479
Name: count, dtype: int64
(1812, 23) mental_risk_category
High        771
Low         562
Moderate    479
Name: count, dtype: int64
Question columns: 21
Demographic columns: []
Missing question columns: []


In [6]:
print("All dataset columns:")
print(df.columns.tolist())

print("\nPossible Age/Gender columns:")
print([
    c for c in df.columns
    if 'age' in c.lower() or 'gender' in c.lower()
])

All dataset columns:
['Q1_1', 'Q1_2', 'Q1_3', 'Q1_4', 'Q1_5', 'Q1_6', 'Q3_1_S1', 'Q3_2_S2', 'Q3_3_S3', 'Q3_4_S4', 'Q3_5_S5', 'Q3_6_S6', 'Q3_7_S7', 'Stress_Score', 'Stress_Level', 'Q3_8_A1', 'Q3_9_A2', 'Q3_10_A3', 'Q3_11_A4', 'Q3_12_A5', 'Q3_13_A6', 'Q3_14_A7', 'Anxiety_Score', 'Anxiety_Level', 'Q3_15_D1', 'Q3_16_D2', 'Q3_17_D3', 'Q3_18_D4', 'Q3_19_D5', 'Q3_20_D6', 'Q3_21_D7', 'Depression_Score', 'Depression_Level', 'stress_score', 'anxiety_score', 'depression_score', 'stress_category', 'anxiety_category', 'depression_category', 'max_severity', 'mental_risk_category']

Possible Age/Gender columns:
[]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print((y_train.value_counts(normalize=True) * 100).round(2))
print((y_test.value_counts(normalize=True) * 100).round(2))

mental_risk_category
High        42.58
Low         30.99
Moderate    26.43
Name: proportion, dtype: float64
mental_risk_category
High        42.42
Low         31.13
Moderate    26.45
Name: proportion, dtype: float64


In [10]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 88.98 %
              precision    recall  f1-score   support

        High       0.93      0.91      0.92       154
         Low       0.94      0.93      0.93       113
    Moderate       0.78      0.81      0.80        96

    accuracy                           0.89       363
   macro avg       0.88      0.88      0.88       363
weighted avg       0.89      0.89      0.89       363

[[140   0  14]
 [  0 105   8]
 [ 11   7  78]]


In [11]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/mental_model.pkl')
joblib.dump(feature_cols, '../models/mental_features.pkl')
print("Saved. Feature order:", feature_cols)

Saved. Feature order: ['Q3_1_S1', 'Q3_2_S2', 'Q3_3_S3', 'Q3_4_S4', 'Q3_5_S5', 'Q3_6_S6', 'Q3_7_S7', 'Q3_8_A1', 'Q3_9_A2', 'Q3_10_A3', 'Q3_11_A4', 'Q3_12_A5', 'Q3_13_A6', 'Q3_14_A7', 'Q3_15_D1', 'Q3_16_D2', 'Q3_17_D3', 'Q3_18_D4', 'Q3_19_D5', 'Q3_20_D6', 'Q3_21_D7', 'Q1_1', 'Q1_2']


In [12]:
import joblib

mental_features = joblib.load('../models/mental_features.pkl')

print("Saved mental model features:")
print(mental_features)

Saved mental model features:
['Q3_1_S1', 'Q3_2_S2', 'Q3_3_S3', 'Q3_4_S4', 'Q3_5_S5', 'Q3_6_S6', 'Q3_7_S7', 'Q3_8_A1', 'Q3_9_A2', 'Q3_10_A3', 'Q3_11_A4', 'Q3_12_A5', 'Q3_13_A6', 'Q3_14_A7', 'Q3_15_D1', 'Q3_16_D2', 'Q3_17_D3', 'Q3_18_D4', 'Q3_19_D5', 'Q3_20_D6', 'Q3_21_D7', 'Q1_1', 'Q1_2']


In [13]:
for i, feature in enumerate(mental_features, start=1):
    print(i, feature)

1 Q3_1_S1
2 Q3_2_S2
3 Q3_3_S3
4 Q3_4_S4
5 Q3_5_S5
6 Q3_6_S6
7 Q3_7_S7
8 Q3_8_A1
9 Q3_9_A2
10 Q3_10_A3
11 Q3_11_A4
12 Q3_12_A5
13 Q3_13_A6
14 Q3_14_A7
15 Q3_15_D1
16 Q3_16_D2
17 Q3_17_D3
18 Q3_18_D4
19 Q3_19_D5
20 Q3_20_D6
21 Q3_21_D7
22 Q1_1
23 Q1_2
